In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../src')

import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import umap


/Users/utkarshmaheshwari/opt/anaconda3/envs/nutrilens/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from data_loading import *
from eda import *
from data_cleaning import *
from k_means import *
from cluster_evaluation import *
from cluster_interpretation import *

In [4]:
embeddings_numpy_arr = np.load("../data/analysis_data/embeddings.npy")
embeddings_df = pd.DataFrame(embeddings_numpy_arr)

In [5]:
path = "../data/analysis_data/only_english.csv"
only_english = read_file(path, seperator=",")
only_english = only_english[['product_name', 'categories_tags', 'nova_group', 'ingredients_text', 'ingredient_count', 'additives_n', 'nutriscore_score', 'monounsaturated-fat_100g','sugars_100g','polyunsaturated-fat_100g','sugar_fiber_ratio','carbohydrates_100g','fiber_100g','saturated-fat_100g','sodium_per_calorie']]
print(only_english.shape)
only_english.head()

Shape of the data: (272809, 18)

Columns loaded:
['code', 'product_name', 'categories_tags', 'ingredients_text', 'ingredient_count', 'additives_n', 'additive_density', 'nutriscore_score', 'monounsaturated-fat_100g', 'sugars_100g', 'polyunsaturated-fat_100g', 'sugar_fiber_ratio', 'carbohydrates_100g', 'fiber_100g', 'saturated-fat_100g', 'sodium_per_calorie', 'allergens_en', 'nova_group']
(272809, 15)


,product_name,categories_tags,nova_group,ingredients_text,ingredient_count,additives_n,nutriscore_score,monounsaturated-fat_100g,sugars_100g,polyunsaturated-fat_100g,sugar_fiber_ratio,carbohydrates_100g,fiber_100g,saturated-fat_100g,sodium_per_calorie
0,Hershey’s Syrup,"en:beverages-and-beverages-preparations,en:bev...",1,organ defat hemp seed powder certifi organ qua...,1,0.0,NaN,0.0,6.0,0.0,0.594059,15.00,10.000,0.0,NaN
1,Vanille-Erdbeer-Cocktail,NaN,4,sunflow seed butter roast sunflow kernel dri c...,30,4.0,NaN,0.0,0.0,0.0,NaN,8.82,0.441,0.0,NaN
2,Naturelo,NaN,4,methylcellulos color preserv flavong ount per ...,5,1.0,NaN,0.0,0.0,0.0,NaN,0.00,0.000,0.0,NaN
3,Prebiotic Fiber Gummies,"en:snacks,en:sweet-snacks,en:confectioneries,e...",4,chicori root fiber water pectin agar agar tapi...,11,6.0,NaN,0.0,0.0,0.0,NaN,0.00,0.000,0.0,NaN
4,Irish Sea Moss,NaN,4,irish sea moss extract chondru crispu hypromel...,3,2.0,NaN,0.0,0.0,0.0,NaN,0.00,0.000,0.0,NaN


In [8]:
df = only_english.copy()
df.shape

(272809, 15)

In [9]:
df2 = df.dropna(axis=0)
df2.shape

(191868, 15)

In [13]:
zero_counts = (df2[['ingredient_count', 'additives_n', 'nutriscore_score',
                           'monounsaturated-fat_100g', 'sugars_100g', 
                           'polyunsaturated-fat_100g', 'sugar_fiber_ratio', 
                           'carbohydrates_100g', 'fiber_100g', 
                           'saturated-fat_100g', 'sodium_per_calorie']] == 0).sum()

print(zero_counts)
print("===================")
print(df2.shape)

ingredient_count                 0
additives_n                  50500
nutriscore_score              7173
monounsaturated-fat_100g    171563
sugars_100g                  28558
polyunsaturated-fat_100g    171122
sugar_fiber_ratio            28558
carbohydrates_100g            6510
fiber_100g                   65648
saturated-fat_100g           66646
sodium_per_calorie           15751
dtype: int64
(191868, 15)


In [ ]:
keep_dist_cat = ["en:baby-foods", "en:fishes-and-their-products", "en:chocolates", "en:salads", "en:ice-creams-and-sorbets", "en:pastas", "en:bodybuilding-supplements", "en:canned-soups", "en:cheeses", "en:sodas"]


def filter_if_all_present(cluster_df, allowed_categories):
    allowed_set = set(allowed_categories)

    def check_row(x):
        if pd.isna(x):
            return False
        cats = str(x).split(',')
        return all(cat.strip() in allowed_set for cat in cats)

    mask = cluster_df['categories_tags'].apply(check_row)
    return cluster_df[mask].copy()

def filter_if_one_present(cluster_df, allowed_categories):
    allowed_set = set(allowed_categories)

    def check_row(x):
        if pd.isna(x):
            return False
        cats = str(x).split(',')
        return any(cat.strip() in allowed_set for cat in cats)

    mask = cluster_df['categories_tags'].apply(check_row)
    return cluster_df[mask].copy()

cluster_df_top_n = filter_if_all_present(df2, top_n_categories)
cluster_df_top_n.shape

Original rows: 191868
Rows remaining after dropping zeros: 8777
